# LangChain 기능 활용

## 1. RetrieverQA 리뷰

In [2]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
!pip install -q --upgrade \
  langchain langchain-core langchain-community langchain-classic \
  langchain-huggingface langchain-ollama \
  chromadb sentence-transformers \
  opentelemetry-api opentelemetry-sdk \
  opentelemetry-exporter-otlp-proto-common opentelemetry-exporter-otlp-proto-grpc

In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


- 프로세스 재시작

In [5]:
!pkill -9 ollama

In [6]:
import subprocess
import time

# 백그라운드로 Ollama 서버 실행
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # 서버 기동 대기
print("Ollama 서버 시작됨 (PID:", process.pid, ")")


Ollama 서버 시작됨 (PID: 14158 )


In [7]:
!ollama --version

ollama version is 0.30.10


- 프로세스 확인

In [8]:
import time
time.sleep(2)
if process.poll() is not None:
    out, err = process.communicate()
    print("프로세스 종료됨! STDOUT:", out)
    print("STDERR:", err)
else:
    print("프로세스 (PID:", process.pid, ")")


프로세스 (PID: 14158 )


- 모형 로딩-최초1회

In [9]:
# 최초 1회 워밍업 - 타임아웃을 넉넉하게(120초) 줘서 로딩이 끝까지 끝나게 둠
!curl -s --max-time 120 http://localhost:11434/api/generate -d '{"model": "qwen2.5:3b", "prompt": "안녕", "stream": false, "options": {"num_predict": 5}. "keep_alive":-1 }'

{"error":"invalid character '.' after object key:value pair"}

In [10]:
!ollama pull qwen2.5:3b

In [11]:
!ollama list

NAME                              ID              SIZE      MODIFIED               
qwen2.5:3b                        357c53fb659c    1.9 GB    Less than a second ago    
engine-diagnosis-expert:latest    d41b1ccf5af8    2.0 GB    38 minutes ago            
llama3.2:3b                       a80c4f17acd5    2.0 GB    41 minutes ago            
gemma:2b                          b50d6c999e59    1.7 GB    42 minutes ago            
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    42 minutes ago            


In [12]:
import requests
import json as json_lib

OLLAMA_URL = "http://localhost:11434/api/generate"

def ask_ollama(model, prompt, temperature=0.3, keep_alive="0s", timeout=120):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature, "num_predict":200,}, #생성 토큰 수 제한
            "keep_alive": "10m",  # 응답 후 즉시 메모리에서 내림            
        },
        timeout=timeout,
    )
    return response.json()["response"]



In [13]:
question = "LLM의 주요 특징을 3개 알려주세요."

model_name = "qwen2.5:3b"
answer = ask_ollama(model_name, question)
print(answer)

LLM(대형 언어 모델)은 다양한 분야에서 중요한 역할을 하고 있으며, 그들에 대한 이해를 높이기 위해 여러 특징들이 있습니다. 여기서는 LLM의 주요 세 가지 특징을 소개하겠습니다:

1. **대량의 훈련 데이터 사용**: LLM들은 수백만 개 이상의 문서와 웹 페이지, 소셜 미디어 게시물 등을 포함하는 대량의 훈련 데이터를 활용합니다. 이로 인해 그들은 매우 넓은 범위의 주제에 대해 이해하고, 다양한 언어 스타일과 문화적 배경을 처리할 수 있습니다.

2. **다양한 언어와 채팅 스타일**: LLM은 여러 언어를 이해하고 사용할 수 있으며, 이는 다양한 문화와 지역의 사용자에게 더욱 유용하게 작동합니다.


In [14]:
#응답시간 오래걸릴 경우 모형 변경:  llama3:8b-instruct-q4_0

modelfile_content = '''
FROM qwen2.5:3b

SYSTEM """
당신은 자동차 엔진부품 제조 공장의 설비 진단 전문가입니다.
다음 원칙을 반드시 지켜 답변하세요.

1. 제공된 정보(context)에 근거해서만 답변하고, 모르면 "제공된 정보로는 알 수 없습니다"라고 답하세요.
2. 4M(Man·Machine·Material·Method) 분류 관점에서 원인을 설명하세요.
3. 답변은 현장 엔지니어가 실제로 참고할 수 있도록 구체적이고 간결하게 작성하세요.
4. 추측이나 일반 상식으로 답변을 지어내지 마세요.
"""

PARAMETER temperature 0.2
PARAMETER top_p 0.9
'''

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print("Modelfile 작성 완료")
print(modelfile_content)


Modelfile 작성 완료

FROM qwen2.5:3b

SYSTEM """
당신은 자동차 엔진부품 제조 공장의 설비 진단 전문가입니다.
다음 원칙을 반드시 지켜 답변하세요.

1. 제공된 정보(context)에 근거해서만 답변하고, 모르면 "제공된 정보로는 알 수 없습니다"라고 답하세요.
2. 4M(Man·Machine·Material·Method) 분류 관점에서 원인을 설명하세요.
3. 답변은 현장 엔지니어가 실제로 참고할 수 있도록 구체적이고 간결하게 작성하세요.
4. 추측이나 일반 상식으로 답변을 지어내지 마세요.
"""

PARAMETER temperature 0.2
PARAMETER top_p 0.9



In [15]:
!ollama create engine-diagnosis-expert -f Modelfile


In [16]:
!ollama list


NAME                              ID              SIZE      MODIFIED               
engine-diagnosis-expert:latest    eadafa4d9310    1.9 GB    Less than a second ago    
qwen2.5:3b                        357c53fb659c    1.9 GB    8 seconds ago             
llama3.2:3b                       a80c4f17acd5    2.0 GB    41 minutes ago            
gemma:2b                          b50d6c999e59    1.7 GB    42 minutes ago            
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    42 minutes ago            


In [17]:
# 커스텀 모델 테스트 - 시스템 프롬프트만으로 역할이 고정되는지 확인
test_question = "치수불량이 발생했을 때 어떤 절차로 점검해야 하나요?"
answer = ask_ollama("engine-diagnosis-expert", test_question)
print(answer)


제공된 정보로는 알 수 없습니다. 그러나 치적 불량이 발생할 경우, 다음과 같은 절차를 통해 원인을 파악하고 문제 해결에 나설 수 있습니다.

1. Man(인력): 우선적으로 해당 부품의 생산 과정에서 인력을 대상으로 작업 방법과 체크 포인트를 확인합니다. 특히 치적 불량이 반복적으로 발생하는 경우, 작업자들의 편집이나 교육 필요성이 있을 수 있습니다.

2. Machine(장비): 장비 상태와 작동을 점검합니다. 치적 불량이 장비의 오작동으로 인한 경우가 많으므로, 장비의 정기적인 유지보수 및 점검이 필요하며, 필요한 경우 교정 또는 변경이 필요할 수 있습니다.

3. Material(자재): 부품의 재료와 제조 과정을 확인합니다. 불


- LangChain + Ollama 연동: QA Chain 백엔드 교체

In [18]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-25 12:33:37--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861 (124K) [text/plain]
Saving to: ‘reports.json.1’

reports.json.1      100%[===================>] 123.89K  --.-KB/s    in 0.003s  

2026-06-25 12:33:38 (45.9 MB/s) - ‘reports.json.1’ saved [126861/126861]



In [19]:
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document          
from langchain_classic.chains import RetrievalQA       
import json

# 데이터 및 벡터스토어 재구성 (새 세션이라면 이 셀부터 실행)
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

#건수 제한(속도)
reports = reports[:150]

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
lc_documents = [
    Document(page_content=r["report_text"], metadata={"report_id": r["report_id"]})
    for r in reports
]
lc_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="lc_engine_reports_ollama",
)
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터스토어 재구성 완료")


/tmp/ipykernel_12505/1742503557.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma as LC_Chroma
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터스토어 재구성 완료


In [20]:
# LLM만 Ollama 커스텀 모델로 교체
ollama_llm = ChatOllama(model="engine-diagnosis-expert", temperature=0.2)

qa_chain_ollama = RetrievalQA.from_chain_type(
    llm=ollama_llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
)

print("Ollama 백엔드 QA Chain 구성 완료")


Ollama 백엔드 QA Chain 구성 완료


In [21]:
question = "포장 관련 불량 사례의 원인은 보통 무엇인가요?"
response = qa_chain_ollama.invoke({"query": question})

print("=== 답변 (Ollama 백엔드) ===")
print(response["result"])
print()
print("=== 참고한 출처 문서 ===")
for doc in response["source_documents"]:
    print("-", doc.metadata["report_id"], ":", doc.page_content[:80])


=== 답변 (Ollama 백엔드) ===
포장 관련 불량 사례의 주된 원인으로는 작업 표준 미흡, 부자재 투입 방향 오류, 포장 속도와 절단 위치 등이 제대로 조절되지 않음, 그리고 제품 전환 시점에서의 문제 등이 있습니다. 이러한 문제가 발생하면 불량품이 생성될 수 있으며 이를 해결하기 위해 작업 표준을 정비하고, 부자재 투입 방향과 포장 속도를 조정하는 등의 조치가 필요합니다.

=== 참고한 출처 문서 ===
- RPT-078 : 2024년 8월 23일 15:45, 자동포장기 B-1호기에서 포장 후 동봉되는 플라스틱 캡과 설명서 방향이 서로 반대로 삽입된 제품을 확인함. 
- RPT-033 : 2024년 4월 7일 10:10, 컨베이어 벨트 3라인에서 플라스틱 커버 이송 후 외관면에 길이 15~28mm의 선형 스크래치가 발생한 것을 확
- RPT-076 : 2024년 8월 17일 09:15, 자동포장기 A-1호기에서 포장 파우치 절단 길이가 기준 180.0mm 대비 176.8~177.5mm로 짧게 


## 2. Agent 방식
### RetrievalQA와 Agent 차이
#### RetrievalQA
- 질문, 검색, LLM, 답변의 흐름이 고정
- 항상 벡터스토어 검색 후, LLM 답변의 순서로만 작동
- Tool 추가 불가하고 구조가 단순하며 속도가 상대적으로 빠름
- 검색 필요없는 질문도 무조건 검색
- RAG만 필요한 경우 사용

#### Agent(create_react_agent)
- 질문, LLM  판단, Tool 선택, 실행, 결과확인, 반복 또는 답변의 순서로 작동
- LLM이 스스로 Tool 선택을 결정(Tool: 특정 기능을 하는 함수)
- Tool을 여러 번 순서 변경하면서 호출 가능
- 구조가 복잡하며 상대적으로 느림.

- RetrievalQA를 AgentExecutor로 변경
    - @tool 데코레이터로 추가할 수 있음
    - PyPDFLoader → 청킹 → 벡터스토어 합산
    - 검색 방식의 변경: search_engine_reports Tool로 래핑
    - 프롬프트: ChatPromptTemplate을 명시적 구성


In [22]:
!pip install langchain-text-splitters -q
!pip install langgraph -q
!pip install pypdf -q
!pip install pdfplumber -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 31.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 118.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 152.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 92.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.



#### 1) 패키지 및 기본 설정

In [23]:

from langchain_ollama import ChatOllama
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent   # ← AgentExecutor 대체
import json
from datetime import datetime

#### 2) 벡터스토어 구성 (reports.json + PDF 통합)

In [24]:

# --- reports.json 로드 ---
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)
reports = reports[:150]

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

lc_documents = [
    Document(
        page_content=r["report_text"],
        metadata={"report_id": r["report_id"], "source": "json"}
    )
    for r in reports
]




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [25]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf

--2026-06-25 12:34:38--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/manual.pdf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1432218 (1.4M) [application/octet-stream]
Saving to: ‘manual.pdf’

manual.pdf          100%[===================>]   1.37M  --.-KB/s    in 0.006s  

2026-06-25 12:34:39 (220 MB/s) - ‘manual.pdf’ saved [1432218/1432218]



In [26]:
# --- PDF 로드 및 청킹 ---
PDF_PATH = "manual.pdf"  # 실제 파일명으로 변경

pdf_loader = PyPDFLoader(PDF_PATH)
pdf_pages  = pdf_loader.load()          # 페이지별 Document 리스트

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pdf_docs = splitter.split_documents(pdf_pages)

# source 메타데이터 부여
for doc in pdf_docs:
    doc.metadata["source"] = "pdf"

print(f"PDF 청크 수: {len(pdf_docs)}")

# --- 통합 벡터스토어 ---
all_documents = lc_documents + pdf_docs

lc_vectorstore = LC_Chroma.from_documents(
    documents=all_documents,
    embedding=lc_embeddings,
    collection_name="lc_engine_reports_with_pdf",
)
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 4})
print("벡터스토어 재구성 완료 (JSON + PDF)")

PDF 청크 수: 14
벡터스토어 재구성 완료 (JSON + PDF)


#### 3) Tool 정의

In [27]:
#docstring이 tool 설명으로 LLM에 전달
@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다. 시간 관련 질문에 사용하세요."""   
    now = datetime.now()
    return now.strftime("현재 시각: %Y년 %m월 %d일 %H시 %M분 %S초")


@tool
def search_engine_reports(query: str) -> str:
    """엔진 고장 및 불량 보고서에서 관련 정보를 검색합니다. 제조 불량 원인 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾지 못했습니다."

    results = []
    for i, doc in enumerate(docs, 1):
        src    = doc.metadata.get("source", "unknown")
        ref_id = doc.metadata.get("report_id", doc.metadata.get("page", ""))
        snippet = doc.page_content[:300]
        results.append(f"[{i}] 출처={src} | ID/페이지={ref_id}\n{snippet}")

    return "\n\n".join(results)


tools = [get_current_time, search_engine_reports]

- bind_tools 사용

In [28]:
ollama_llm = ChatOllama(model="qwen2.5:3b", temperature=0.2)
llm_with_tools = ollama_llm.bind_tools(tools)

#LLM에게 질문 → tool_calls 반환 (실행 안 함)
question = "포장 관련 불량 원인은?"
response = llm_with_tools.invoke(question)

print("=== LLM 응답 ===")
print(response.content)
print("\n=== Tool 호출 의도 ===")
print(response.tool_calls)   # [{'name': 'search_engine_reports', 'args': {...}}]

#tool_calls 보고 개발자가 직접 실행
tool_map = {t.name: t for t in tools}

results = []
for tc in response.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    print(f"\n실행: {tool_name}({tool_args})")
    
    result = tool_map[tool_name].invoke(tool_args)  # 여기서 실제 실행
    print(f"결과: {result[:200]}")
    results.append(result)



=== LLM 응답 ===


=== Tool 호출 의도 ===
[{'name': 'search_engine_reports', 'args': {'query': '포장 관련 불량 원인'}, 'id': '5dd60da6-62f3-4d32-9d0a-7ad034eb4450', 'type': 'tool_call'}]

실행: search_engine_reports({'query': '포장 관련 불량 원인'})
결과: [1] 출처=json | ID/페이지=RPT-078
2024년 8월 23일 15:45, 자동포장기 B-1호기에서 포장 후 동봉되는 플라스틱 캡과 설명서 방향이 서로 반대로 삽입된 제품을 확인함. 포장속도는 110포/분이었고 부자재 투입 방향 표준 이미지가 작업대에 없어 포장팀에서 심각도 하로 기록함. 부자재 투입 방향에 대한 작업표준이 미흡해 조립 방향 오


In [29]:
# 결과를 다시 LLM에 넣어 최종 답변 생성
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(content=question),
    response,  # LLM의 tool_calls 포함된 응답
    *[
        ToolMessage(content=r, tool_call_id=tc["id"])
        for r, tc in zip(results, response.tool_calls)
    ]
]

final_response = llm_with_tools.invoke(messages)
print("\n=== 최종 답변 ===")
print(final_response.content)


=== 최종 답변 ===
포장 관련 불량 원인에 대해 다음과 같은 보고서가 있습니다:

1. **2024년 8월 23일** : 포장 후 동봉되는 플라스틱 캡과 설명서 방향이 서로 반대로 삽입된 제품을 확인했습니다. 이는 부자재 투입 방향 표준 이미지가 작업대에 없어 발생한 것으로 보입니다. 이를 해결하기 위해, 선별하여 재포장하고 부자재 투입구에 방향 기준 이미지를 부착하였습니다.

2. **2024년 4월 7일** : 컨베이어 벨트 3라인에서 플라스틱 커버 이송 후 외관면에 길이 15~28mm의 선형 스크래치가 발생했습니다. 이는 완충지가 누락된 묶음에서 결함이 집중되어 발생한 것으로 판단되었습니다. 이를 해결하기 위해, 완충지 누락 자재를 격리하고 외관 선별을 실시했으며 공급업체에 포장 상태 개선과 재발 방지를 위한 보고서를 요청하였습니다.

3. **2024년 4월 13일** : 컨베이어 벨트 5라인에서 액상 접착제 카트리지 이송 중 6개 제품의 마개 주변 누설을 확인했습니다. 이는 입고 카트리지 캡 내경이 규격보다 크게 측정되어 밀봉력 부족으로 인해 발생한 것으로 보입니다. 이를 해결하기 위해, 해당 로트의 캡을 전량 격리하고 규격품 캡으로 교체했으며 누설품은 외관과 중량을 확인한 뒤 재작업 여부를 판정하였습니다.

4. **2024년 5월 11일** : 컨베이어 벨트 13라인의 검사 대기 박스 내부에서 5~12mm 크기의 투명 필름 조각이 발견되었습니다. 이는 보호필름 절단 상태 불량으로 인해 발생한 것으로 보입니다. 이를 해결하기 위해, 해당 원자재 로트를 격리하고 보호필름 박리 검사를 실시했으며 공급업체에 절단 공정 개선과 대체 로트 납품을 요청하였습니다.

이러한 불량 원인들은 대부분 부자재 품질 문제나 포장 방식의 오류로 인해 발생한 것으로 보입니다. 이를 해결하기 위해, 작업 표준 정비와 공급업체에 대한 지시사항 제공 등이 필요할 것입니다.


#### 4) Agent 구성

In [30]:
ollama_llm = ChatOllama(model="engine-diagnosis-expert", temperature=0.2, timeout=120, num_predict=200)

agent_executor = create_react_agent(
    model=ollama_llm,
    tools=tools,
    prompt="당신은 제조업 엔진 진단 전문가입니다. "
           "질문에 답하기 위해 필요한 도구를 사용하세요. "
           "검색 결과를 바탕으로 근거 있는 답변을 제공하고, 출처를 명시하세요.",
)
print("Agent 구성 완료")


Agent 구성 완료


/tmp/ipykernel_12505/984660702.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


#### 5) 실행 테스트

In [31]:
questions = [
    "지금 몇 시야?",
    "포장 관련 불량 사례의 원인은 보통 무엇인가요?",
    "지금 시각과 함께 최근 포장 불량 사례를 요약해줘",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"질문: {q}")
    print('='*60)
    result = agent_executor.invoke({"messages": [{"role": "user", "content": q}]})
    # 마지막 메시지가 최종 답변
    print("\n[최종 답변]")
    print(result["messages"][-1].content)


질문: 지금 몇 시야?

[최종 답변]
현재 시간은 2026년 06월 25일, 12시 34분 46초입니다.

질문: 포장 관련 불량 사례의 원인은 보통 무엇인가요?

[최종 답변]
<JSONObject>
"name": "search_engine_reports",
"arguments": {
  "query": "제조 포장 불량 원인"
}
</JSONObject>

이 질문에 대한 정보를 찾기 위해 엔진 고장 및 불량 보고서에서 관련 정보를 검색하겠습니다.

질문: 지금 시각과 함께 최근 포장 불량 사례를 요약해줘

[최종 답변]
최근에 발생한 포장 불량 사례를 요약해 드리겠습니다:

1. **2024년 8월 23일** - 자동포장기 B-1호기에서 플라스틱 캡과 설명서 방향이 반대로 삽입된 제품이 발생했습니다. 이는 부자재 투입 방향 표준 이미지가 없어 심각도 하로 분류되었습니다.

2. **2024년 4월 7일** - 컨베이어 벨트 3라인에서 플라스틱 커버 이송 후 외관면에 선형 스크래치가 발생했습니다. 이는 완충지 누락으로 인해 포장팀에서 심각도 하로 분류되었습니다.

3. **2024년 11월 25일**


## 3. 인터넷 검색 후 답변하는 랭체인

#### 덕덕고 설치
- 랭체인은 구글검색, 빙, 덕덕고, 타일리 등 활용 가능
- 덕덕고 무료 API 활용

In [32]:
!pip install ddgs langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 141.2 MB/s eta 0:00:0000:01


In [33]:
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 검색 툴
search = DuckDuckGoSearchRun()

# Agent
llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, timeout=120)

agent = create_react_agent(
    model=llm,
    tools=[search],
    prompt="당신은 인터넷 검색을 통해 최신 정보를 제공하는 assistant입니다. 항상 검색 후 답변하세요.",
)

# 실행
result = agent.invoke({"messages": [{"role": "user", "content": "2026년 최신 AI 뉴스는?"}]})
print(result["messages"][-1].content)

/tmp/ipykernel_12505/555745687.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


Here are some of the latest news related to AI from the sources I found:

1. **Apple**: There will be a reveal event featuring new tools, frameworks, and features for apps and games, with video sessions hosted by Apple engineers and designers.

2. **Google**: News about Google products, technology, and innovation can be found on their official blog.

3. **The Guardian**: The Guardian provides the latest news, sports, business, comments, analysis, and reviews.

4. **Anthropic’s Claude AI Platform**: There was a significant service disruption affecting multiple flagship models of Anthropic's Claude AI platform, which lasted for nearly 90 minutes before engineers restored full functionality on June 22, 2026.

Would you like more details or specific information from any of these sources?


### RAG+인터넷 검색

In [34]:
from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import json

#### 1) 벡터스토어 구성 (기존과 동일)

In [35]:
with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)
reports = reports[:150]

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
lc_documents = [
    Document(page_content=r["report_text"], metadata={"report_id": r["report_id"]})
    for r in reports
]
lc_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="lc_engine_reports",
)
retriever = lc_vectorstore.as_retriever(search_kwargs={"k": 3})
print("벡터스토어 준비 완료")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

벡터스토어 준비 완료


#### 2) Tool 정의 - RAG + 검색

In [36]:
duckduckgo = DuckDuckGoSearchRun()

@tool
def search_reports(query: str) -> str:
    """사내 엔진 고장/불량 보고서에서 검색합니다. 내부 데이터, 과거 사례, 불량 원인 질문에 사용하세요."""
    docs = retriever.invoke(query)
    if not docs:
        return "관련 보고서 없음"
    results = []
    for doc in docs:
        results.append(f"[{doc.metadata['report_id']}] {doc.page_content[:300]}")
    return "\n\n".join(results)

@tool
def search_internet(query: str) -> str:
    """인터넷에서 최신 정보를 검색합니다. 최신 뉴스, 외부 기술 정보, 일반 지식 질문에 사용하세요."""
    return duckduckgo.invoke(query)

tools = [search_reports, search_internet]

#### 3) Agent

In [37]:
# Agent
llm = ChatOllama(model="qwen2.5:3b", temperature=0.2, timeout=120)

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=(
        "당신은 제조업 엔진 진단 전문가입니다.\n"
        "- 사내 보고서 관련 질문 → search_reports 사용\n"
        "- 최신 정보/외부 기술 질문 → search_internet 사용\n"
        "- 필요시 두 Tool을 모두 사용해 종합 답변하세요."
    ),
)

/tmp/ipykernel_12505/4210200781.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


#### 4) 실행

In [38]:
questions = [
    "포장 불량의 주요 원인은?",           # RAG
    "2026년 제조업 AI 트렌드는?",         # 인터넷
    "포장 불량 사례와 최신 해결 기술은?",  # 둘 다
]

for q in questions:
    print(f"\n{'='*60}\n질문: {q}\n{'='*60}")
    result = agent.invoke({"messages": [{"role": "user", "content": q}]})
    print(result["messages"][-1].content)


질문: 포장 불량의 주요 원인은?
포장 불량의 주요 원인은 다음과 같습니다:

1. **부식성 물질**: 부식이 발생하여 포장재가 손상되는 경우가 있습니다.
2. **외부 물질 contamination**: 포장 과정에서 외부에 들어온 물질로 인한 contamination 문제입니다.
3. **압력 부족**: 압력 부족으로 인해 냉동식품이나 음료의 포장이 손상되는 경우가 있습니다.
4. **설계 및 제조 오류**: 포장 설계나 제조 과정에서 발생하는 오류로 인한 문제입니다.
5. **설계 및 제조 품질 불량**: 부품이나 재료의 품질이 저급하거나 부적합하여 포장을 손상시키는 경우가 있습니다.

포장 불량을 방지하기 위해서는 다음과 같은 조치를 취해야 합니다:

- 정확한 설계와 제조
- 적절한 재료 선택과 품질 검사
- 포장 후의 관리 및 보관 방법

이러한 문제들은 대부분 포장 과정에서 발생하며, 이를 해결하기 위해서는 포장 설계, 제조, 품질 관리 등 다양한 측면을 고려해야 합니다. 또한, 포장 재료와 설계에 대한 지속적인 연구 및 개선이 필요합니다.

질문: 2026년 제조업 AI 트렌드는?
Based on the information found:

- There is a growing interest in how AI can reduce costs and improve operations.
- The focus is shifting from initial excitement to a more mature understanding of what AI can achieve in manufacturing.
- In 2026, manufacturers will likely rely heavily on AI-enabled energy management systems, including agentic control and real-time flow optimization.

These trends suggest that by 2026, AI will be pl